In [ ]:
#| default_exp diagnostics

# API diagnostics and project paths

> Record tool usage, resolve project inputs and bound diagnostic output.

These operations own process-level diagnostics and path discovery. The notebook data model uses them through imports; diagnostic code does not depend on that model.

In [ ]:
from nbskill.fixtures import demo_path, write_demo_file, write_demo_notebook
from io import StringIO
from tempfile import TemporaryDirectory
from contextlib import redirect_stderr
from fastcore.test import test_fail

In [ ]:
#| hide
from inspect import signature
from unittest.mock import patch as patch_env

In [ ]:
#| export
import hashlib, json, logging, os, re, subprocess, tempfile, time
from contextvars import ContextVar
from functools import wraps
from pathlib import Path
from fastcore.basics import AttrDict
from nbskill.fixtures import path_candidates

#| export
## Python API contracts and diagnostics

#| export
The public functions in later notebooks are direct Python APIs. These helpers keep return values and validation errors predictable while recording completed calls and failures in a small local statistics file.

`api_return` preserves the Python value and its identity for direct callers.

In [ ]:
#| export
def api_return(value=None):
    "Return a Python API value unchanged."
    return value

In [ ]:
payload = dict(answer=42)
returned = api_return(payload)
assert returned is payload
returned

Direct API calls return Python values. `api_error` raises `ValueError` for invalid inputs; callers retain the exception rather than receiving an error-shaped success value.

`api_error` raises a validation error so callers retain its original message.

In [ ]:
#| export
def api_error(msg):
    "Raise a Python API validation error."
    raise ValueError(msg)

In [ ]:
try: api_error("a notebook path is required")
except ValueError as error: validation_message = str(error)
assert validation_message == "a notebook path is required"
validation_message

In [ ]:
#| hide
test_fail(lambda: api_error("path required"), contains="path required")

In [ ]:
#| export
def failure_map_path():
    'Resolve the usage-statistics path from the environment or home directory.'
    default = Path.home() / ".nbskill-errors.json"
    return Path(os.environ.get("NBSKILL_FAILURE_MAP", default)).expanduser()

`failure_map_path` honors `NBSKILL_FAILURE_MAP` for isolated runs. `empty_failure_map` creates a fresh versioned record with no calls or events; it does not create a file.

Each call to `empty_failure_map` starts an independent event history.

In [ ]:
#| export
def empty_failure_map():
    'Return a fresh empty usage-statistics record.'
    return dict(version=1, events=[], counts={}, last_call=None)

In [ ]:
first_history = empty_failure_map()
first_history["events"].append(dict(operation="context"))
assert empty_failure_map()["events"] == []

In [ ]:
#| export
def load_failure_map(path, strict=False):
    "Read saved usage, optionally rejecting corrupt or unreadable statistics."
    try: data = json.loads(path.read_text(encoding="utf-8"))
    except FileNotFoundError: data = empty_failure_map()
    except (json.JSONDecodeError, OSError):
        if strict: raise
        data = empty_failure_map()
    if not isinstance(data, dict): raise ValueError("nbskill usage must be a JSON object")
    data.setdefault("version", 1)
    data.setdefault("events", [])
    data.setdefault("counts", {})
    data.setdefault("last_call", None)
    return data

`load_failure_map` supplies empty counts for a missing file. Writers use `strict=True` to reject corrupt JSON and preserve the original file for diagnosis. The default read mode still tolerates missing or unreadable statistics.

In [ ]:
empty_usage = load_failure_map(demo_path("missing_usage.json"), strict=True)
empty_usage

In [ ]:
#| hide
with tempfile.TemporaryDirectory() as usage_dir:
    invalid_usage = Path(usage_dir)/"usage.json"
    invalid_usage.write_text("{broken")
    try: load_failure_map(invalid_usage, strict=True)
    except json.JSONDecodeError: pass
    else: raise AssertionError("Strict reads must reject corrupt usage JSON")
    assert invalid_usage.read_text() == "{broken"

In [ ]:
#| exporti
#| hide
_ACTIVE_USAGE_CALL = ContextVar("nbskill_usage_call", default=None)
_USAGE_EVENT_LIMIT = 100
_USAGE_ERROR_LIMIT = 1000

In [ ]:
#| exporti
#| hide
def _update_usage(data, tool, error=None):
    counts, events = data["counts"], data["events"]
    if not isinstance(counts, dict) or not isinstance(events, list): raise ValueError("Invalid nbskill usage counts or events")
    groups = ["usage"] if error is None else ["usage", "failures"]
    for group in groups:
        totals = counts.setdefault(group, {})
        if not isinstance(totals, dict): raise ValueError(f"Invalid {group} counts")
        count = totals.get(tool, 0)
        if not isinstance(count, int) or count < 0: raise ValueError(f"Invalid {tool} count")
        totals[tool] = count + 1
    event = dict(tool=tool, cwd=str(Path.cwd()), ts=time.time())
    data["last_call"] = event
    if error is not None:
        events.append(dict(kind="failure", error=error[:_USAGE_ERROR_LIMIT], **event))
        data["events"] = events[-_USAGE_EVENT_LIMIT:]

Usage counters and failures are committed together. The writer replaces the statistics file atomically after the outermost tracked call; readers never need to merge partially written records.

In [ ]:
#| exporti
#| hide
def _write_usage(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    handle = tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent, delete=False)
    temporary = Path(handle.name)
    try:
        with handle: json.dump(data, handle, indent=2, sort_keys=True)
        os.replace(temporary, path)
    finally: temporary.unlink(missing_ok=True)

In [ ]:
#| exporti
#| hide
def _record_usage_call(tool, error=None):
    if _ACTIVE_USAGE_CALL.get() != os.getpid(): return
    from nbskill.parallel import notebook_locks
    path = failure_map_path()
    try:
        with notebook_locks(path):
            data = load_failure_map(path, strict=True)
            _update_usage(data, tool, error)
            _write_usage(path, data)
    except (OSError, ValueError, TypeError) as exc: logging.getLogger(__name__).warning("Could not record nbskill %s usage: %s", tool, exc)

`track_call` restores recording at the direct Python API boundary after the old command wrappers were removed. The eight Pyskill operations and `style_report` use it in their owning modules, so direct imports and Pyskill imports count the same calls. `NBSKILL_FAILURE_MAP` selects a different statistics file; the default remains `~/.nbskill-errors.json`.

Each completed or interrupted outermost call increments `counts.usage` and updates `last_call` with its tool name, working directory and completion time. Usage and failure counts are committed together. Nested tool calls in the same process are excluded. Notebook execution children count their own calls. Ordinary exceptions and results with `ok=False` also increment `counts.failures`; the original exception or result reaches the caller unchanged. Cancellation releases the call scope. Function signatures, docments and return identities are preserved.

The existing path locks serialize threads and processes. A temporary file and atomic replacement keep readers from seeing partial JSON. Corrupt statistics are preserved for diagnosis, and persistence failures produce a logging warning while the tool continues. Failure details are capped at 1000 characters and the most recent 100 events are kept. Counts remain cumulative until an explicit reset. Resetting also discards the current call's pending record.

In [ ]:
#| export
def track_call(func):
    "Record an outermost synchronous tool call without changing its Python contract."
    @wraps(func)
    def tracked(*args, **kwargs):
        if _ACTIVE_USAGE_CALL.get() == os.getpid(): return func(*args, **kwargs)
        token = _ACTIVE_USAGE_CALL.set(os.getpid())
        try: result = func(*args, **kwargs)
        except BaseException as exc:
            error = f"{type(exc).__name__}: {exc}" if isinstance(exc, Exception) else None
            _record_usage_call(func.__name__, error)
            raise
        else:
            error = None
            if isinstance(result, dict) and result.get("ok") is False: error = str(result.get("errors") or result.get("text") or "ok=False")
            _record_usage_call(func.__name__, error)
            return result
        finally: _ACTIVE_USAGE_CALL.reset(token)
    return tracked

A tracked calculation returns its value while writing one usage count. The temporary destination keeps lesson calls out of the global statistics.

In [ ]:
with tempfile.TemporaryDirectory() as usage_dir, patch_env.dict(os.environ, NBSKILL_FAILURE_MAP=str(Path(usage_dir)/"usage.json")):
    tracked_add = track_call(lambda left, right: left + right)
    added_cells = tracked_add(2, 3)
    example_usage = load_failure_map(failure_map_path())
added_cells, example_usage["counts"]

In [ ]:
#| hide
with tempfile.TemporaryDirectory() as usage_dir, patch_env.dict(os.environ, NBSKILL_FAILURE_MAP=str(Path(usage_dir)/"usage.json")):
    payload = object()
    transparent_return = track_call(api_return)
    assert track_call(api_return)(payload) is payload
    assert (signature(transparent_return), transparent_return.__doc__) == (signature(api_return), api_return.__doc__)
    assert load_failure_map(failure_map_path())["counts"]["usage"] == {"api_return": 1}

In [ ]:
#| hide
with tempfile.TemporaryDirectory() as usage_dir, patch_env.dict(os.environ, NBSKILL_FAILURE_MAP=str(Path(usage_dir)/"usage.json")):
    inner = track_call(api_return)
    outer = track_call(lambda: inner("nested result"))
    assert outer() == "nested result"
    assert load_failure_map(failure_map_path())["counts"]["usage"] == {"<lambda>": 1}

In [ ]:
#| hide
with tempfile.TemporaryDirectory() as usage_dir, patch_env.dict(os.environ, NBSKILL_FAILURE_MAP=str(Path(usage_dir)/"usage.json")):
    expected_error = ValueError("invalid notebook")
    failed_result = dict(ok=False, errors=["cell failed"])
    @track_call
    def reject(): raise expected_error
    @track_call
    def execute_failure(): return failed_result
    @track_call
    def cancel(): raise KeyboardInterrupt()
    try: reject()
    except ValueError as exc: assert exc is expected_error
    else: raise AssertionError("The original exception should reach the caller")
    assert execute_failure() is failed_result
    try: cancel()
    except KeyboardInterrupt: pass
    else: raise AssertionError("Cancellation should reach the caller")
    execute_failure()
    data = load_failure_map(failure_map_path())
    assert data["counts"] == dict(usage=dict(reject=1, execute_failure=2, cancel=1), failures={"reject": 1, "execute_failure": 2})

In [ ]:
#| hide
with tempfile.TemporaryDirectory() as usage_dir, patch_env.dict(os.environ, NBSKILL_FAILURE_MAP=str(Path(usage_dir)/"usage.json")):
    path = failure_map_path()
    path.write_text("{broken")
    @track_call
    def usable(): return "still usable"
    with redirect_stderr(StringIO()) as diagnostics: assert usable() == "still usable"
    assert path.read_text() == "{broken"
    assert "Could not record nbskill" in diagnostics.getvalue()


In [ ]:
#| hide
with tempfile.TemporaryDirectory() as usage_dir, patch_env.dict(os.environ, NBSKILL_FAILURE_MAP=str(Path(usage_dir)/"usage.json")):
    failure_map_path().mkdir()
    with redirect_stderr(StringIO()) as diagnostics: assert usable() == "still usable"
    assert "Could not record nbskill" in diagnostics.getvalue()

In [ ]:
#| hide
with tempfile.TemporaryDirectory() as usage_dir, patch_env.dict(os.environ, NBSKILL_FAILURE_MAP=str(Path(usage_dir)/"usage.json")):
    @track_call
    def failed(): return dict(ok=False, errors=["x" * 2000])
    for _ in range(105): failed()
    data = load_failure_map(failure_map_path())
    assert (data["counts"]["usage"]["failed"], data["counts"]["failures"]["failed"]) == (105, 105)
    assert len(data["events"]) == 100
    assert all(len(event["error"]) <= 1000 for event in data["events"])

`reset_usage` clears the file under the writer lock and discards the active call's pending record. This preserves `style_check(delete_after_output=True)` without recreating the file after printing.

In [ ]:
#| export
def reset_usage(path=None):
    "Clear selected statistics and discard the current call's pending record."
    from nbskill.parallel import notebook_locks
    path = failure_map_path() if path is None else Path(path).expanduser()
    with notebook_locks(path): path.unlink(missing_ok=True)
    _ACTIVE_USAGE_CALL.set(None)

In [ ]:
with tempfile.TemporaryDirectory() as usage_dir, patch_env.dict(os.environ, NBSKILL_FAILURE_MAP=str(Path(usage_dir)/"usage.json")):
    failure_map_path().write_text(json.dumps(empty_failure_map()))
    reset_usage()
    cleared_usage = not failure_map_path().exists()
cleared_usage

In [ ]:
#| hide
with tempfile.TemporaryDirectory() as usage_dir, patch_env.dict(os.environ, NBSKILL_FAILURE_MAP=str(Path(usage_dir)/"usage.json")):
    read_counts = track_call(api_return)
    read_counts()
    assert reset_usage() is None and not failure_map_path().exists()
    read_counts()
    assert load_failure_map(failure_map_path())["counts"]["usage"] == {"api_return": 1}
    clear_counts = track_call(reset_usage)
    clear_counts()
    assert not failure_map_path().exists()

Git discovery is optional. `git_run` returns empty text for a failed command, and `git_root` returns `None` when no candidate belongs to a repository. Source inspection can still work outside Git.

Repository discovery only reads `.git` markers, including the files used by worktrees and submodules. Launching `git rev-parse` here made ordinary notebook reads require subprocess permission in the sandbox.

`git_run` returns command output for repository inspection.

In [ ]:
#| export
def git_run(root, *args):
    "Run a git command from `root` and return stdout, or empty text on failure."
    proc = subprocess.run(["git", "-C", str(root), *args], text=True, capture_output=True)
    return proc.stdout if proc.returncode == 0 else ""

In [ ]:
inside_repository = git_run(Path.cwd(), "rev-parse", "--is-inside-work-tree").strip()
assert inside_repository == "true"
inside_repository

`git_root` resolves the repository containing the current working directory.

In [ ]:
#| export
def git_root(path="."):
    "Return the containing git repository root, if one exists."
    path = Path(path).expanduser()
    base = path.parent if path.suffix else path
    candidates = [base, *path_candidates(base), Path(".")]
    for candidate in dict.fromkeys(candidates):
        candidate = candidate.resolve()
        for root in (candidate, *candidate.parents):
            if (root/".git").is_dir() or (root/".git").is_file(): return root
    return None

In [ ]:
repository = git_root(Path.cwd())
assert repository is not None and (repository/".git").exists()
repository.name

In [ ]:
#| exporti
#| hide
def _is_notebook_path(path):
    path = Path(path)
    hidden = any(part.startswith(".") or part == ".ipynb_checkpoints" for part in path.parts)
    return path.suffix == ".ipynb" and not hidden and not path.name.startswith("_")

In [ ]:
#| exporti
#| hide
def _has_glob_chars(path): return any(char in str(path) for char in "*?[]")

`load_nbskill_config` reads trusted project Python from `.nbskill.py` and overlays its public names on the defaults. This configuration file executes with the agent process permissions.

`load_nbskill_config` merges project settings over the default notebook exclusions.

In [ ]:
#| export
_DEFAULT_CONFIG = AttrDict(exclude_dirs=["data", "dev", ".ipynb_checkpoints"])

def load_nbskill_config(path="."):
    "Load .nbskill.py config from project root, merged over defaults."
    cfg = AttrDict(**_DEFAULT_CONFIG)
    root = git_root(path)
    if root is None: root = Path(path).resolve()
    cfg_path = Path(root) / ".nbskill.py"
    if cfg_path.exists():
        ns = {}
        exec(cfg_path.read_text(encoding="utf-8"), ns)
        cfg.update({k: v for k, v in ns.items() if not k.startswith("_")})
    return cfg

In [ ]:
configuration = load_nbskill_config(".")
assert ".ipynb_checkpoints" in configuration.exclude_dirs
configuration.exclude_dirs

In [ ]:
#| exporti
#| hide
def _glob_notebook_paths(path):
    try:
        from fastcore.xtras import globtastic
        cfg = load_nbskill_config(path)
        skip_re = r"(^[_.]|" + "|".join(re.escape(d) for d in cfg.exclude_dirs) + r")"
        if _has_glob_chars(path):
            import glob
            return [Path(item) for item in glob.glob(str(path), recursive=True)]
        return [Path(item) for item in globtastic(path, file_glob="*.ipynb", skip_folder_re=skip_re, skip_file_re=r"^[_.]", func=Path)]
    except Exception: return sorted(Path(path).rglob("*.ipynb")) if Path(path).is_dir() else []

In [ ]:
#| exporti
#| hide
def _nbdev_notebook_paths(path):
    try:
        from nbdev.doclinks import nbglob
        return [Path(item) for item in nbglob(path=path, as_path=True)]
    except Exception: return []

Metadata migration removes obsolete stored analysis fields because source properties are recomputed from current code.

In [ ]:
#| export
def notebook_paths(path="nbs"):
    "Return visible notebook paths for a file, directory, glob, or nbdev project path."
    raw = "." if path is None else str(path)
    cfg = load_nbskill_config(raw)
    exclude = set(cfg.exclude_dirs)
    base = Path(raw).expanduser().resolve()
    def _excluded(p):
        p = Path(p).expanduser()
        if not p.is_absolute(): p = p.resolve() if p.exists() else (base / p).resolve()
        try: rel = p.relative_to(base)
        except ValueError: return False
        return any(part in exclude for part in rel.parts)
    for candidate in path_candidates(raw):
        pth = Path(candidate).expanduser()
        if pth.is_file(): return [pth] if _is_notebook_path(pth.name) else []
        elif _has_glob_chars(raw): paths = _glob_notebook_paths(pth)
        elif pth.is_dir(): paths = _nbdev_notebook_paths(pth) or _glob_notebook_paths(pth)
        else: paths = []
        paths = sorted({item for item in paths if _is_notebook_path(item) and not _excluded(item)})
        if paths: return paths
    return []

`notebook_paths` keeps an explicitly selected notebook reachable inside a hidden workspace folder such as `.codex`. It checks the filename in this case; directory discovery still filters hidden paths. A temporary hidden folder demonstrates the explicit-file contract.

In [ ]:
visible_notebooks = notebook_paths("nbs/00_foundation.ipynb")
visible_notebooks

In [ ]:
#| hide
with TemporaryDirectory(prefix=".nbskill-") as folder:
    with write_demo_notebook("selected.ipynb", base=folder) as path: assert notebook_paths(path) == [path]

Python analysis uses `source_without_directives` to remove notebook control lines.

In [ ]:
#| export
def source_without_directives(source):
    "Return source with nbdev cell directives removed."
    return "\n".join(line for line in str(source or "").splitlines() if not line.lstrip().startswith("#|"))

In [ ]:
plain_source = source_without_directives("#| export\nanswer = 42")
assert plain_source == "answer = 42"
plain_source

`source_without_directives` removes notebook control lines before Python analysis. `source_hash` fingerprints the supplied text exactly; directive removal is an explicit caller decision.

`source_hash` compares source content without storing another editable copy.

In [ ]:
#| export
def source_hash(source, length=12):
    'Return a SHA-256 source hash, shortened unless `length` is `None`.'
    digest = hashlib.sha256(str(source).encode('utf-8')).hexdigest()
    return digest if length is None else digest[:int(length)]

In [ ]:
assert source_hash("answer = 42") == source_hash("answer = 42")
assert source_hash("answer = 42") != source_hash("answer = 43")

`file_line_count` counts physical lines in an existing source file.

In [ ]:
#| export
def file_line_count(path):
    "Return the number of text lines in path, or zero when it cannot be read."
    try: return len(Path(path).read_text(encoding="utf-8", errors="ignore").splitlines())
    except OSError: return 0

In [ ]:
with write_demo_file("two_lines.py", "first = 1\nsecond = 2\n") as two_lines: assert file_line_count(two_lines) == 2

`file_line_count` treats an unreadable file as unavailable size evidence. `cap_text` bounds display text and, when requested, reports omitted characters without discarding the underlying structured result.

`cap_text` bounds the rendered text while recording whether content was omitted.

In [ ]:
#| export
def cap_text(text, limit=2000, max_output_chars=...):
    "Bound the complete text; an explicit max_output_chars returns metadata, with None meaning unlimited."
    metadata = max_output_chars is not ...
    limit = max_output_chars if metadata else limit
    if limit is not None and limit < 0: raise ValueError("text budget must be non-negative or None")
    text = "" if text is None else str(text)
    if limit is None or len(text) <= limit: capped, omitted = text, 0
    else:
        marker = "\n... truncated ..."
        content_limit = limit - len(marker) if limit >= len(marker) else limit
        prefix = text[:content_limit].rstrip()
        capped = prefix + marker if limit >= len(marker) else prefix
        omitted = len(text) - len(prefix)
    if not metadata: return capped
    return dict(text=capped, truncated=bool(omitted), chars=len(text), omitted_chars=omitted)

In [ ]:
bounded_example = cap_text("a long result", max_output_chars=8)
assert len(bounded_example["text"]) <= 8 and bounded_example["truncated"]
bounded_example

`NotebookResult` retains dictionary access while limiting what an interpreter displays. The complete result remains available for subsequent operations.

In [ ]:
#| export
class NotebookResult(dict):
    "Dictionary result with one compact interpreter display."
    def __repr__(self): return self["display"] if "display" in self else dict.__repr__(self)

In [ ]:
#| export
def notebook_result(data, max_display_chars=6000):
    "Bound the interpreter display while retaining the complete structured text."
    capped = cap_text(data.get("text", ""), max_output_chars=max_display_chars)
    return NotebookResult({**data, "display": capped["text"], "display_budget": capped})

`notebook_result` bounds the interpreter display while retaining the complete structured result for callers.

In [ ]:
result_example = notebook_result(dict(text="complete evidence " * 50), max_display_chars=80)
assert len(repr(result_example)) <= 80
assert len(result_example["text"]) > 80
repr(result_example)

Display limits include the truncation marker itself. Structured results retain the full text, and an explicit None budget keeps it unabridged.

In [ ]:
#| hide
long_text = "visible evidence " * 100
for budget in (0, 8, 80):
    bounded = cap_text(long_text, max_output_chars=budget)
    assert len(bounded["text"]) <= budget and bounded["truncated"]
    assert len(repr(notebook_result(dict(text=long_text), max_display_chars=budget))) <= budget

In [ ]:
#| hide
unbounded = cap_text(long_text, max_output_chars=None)
assert unbounded["text"] == long_text and not unbounded["truncated"]
assert repr(notebook_result(dict(text=long_text), max_display_chars=None)) == long_text